# RocksDB WAL: 5-Study Verification Suite
---
This notebook provides a data-driven verification of all five engineering studies performed on the instrumented RocksDB Write-Ahead Log.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

os.makedirs('./docs/images', exist_ok=True)
sns.set_theme(style="whitegrid")
df = pd.read_csv('./results/wal_performance_telemetry.csv')
df.head()

## Study 1: The Safety Tax (Latency)
Comparing Buffered vs Strict Sync modes.

In [ ]:
batch = df[df['Event'] == 'WAL_Batch']
plt.figure(figsize=(10, 5))
sns.barplot(x=['Buffered', 'No-WAL', 'Strict Sync'], y=batch['Metric'].values[:3] / 1000)
plt.title('Ingestion Latency per MODE (\u03bcs)')
plt.savefig('./docs/images/exp1_throughput.png', bbox_inches='tight')
plt.show()

## Study 2: Fragmentation Efficiency
Ratio of payload data to header overhead.

In [ ]:
overhead = df[df['Event'].str.contains('WAL_Bytes')]
plt.figure(figsize=(6, 6))
plt.pie(overhead['Metric'], labels=['Headers', 'Payload'], autopct='%1.1f%%', colors=['#e74c3c', '#2ecc71'])
plt.title('Internal Space Utilization')
plt.savefig('./docs/images/exp2_fragmentation.png', bbox_inches='tight')
plt.show()

## Study 3: Recovery Mode Performance
Impact of WALRecoveryMode on startup time.

In [ ]:
rec_modes = df[df['Event'] == 'WAL_Recovery_Mode']
plt.figure(figsize=(10, 5))
sns.barplot(x=rec_modes['Count'], y=rec_modes['Metric'].astype(int), palette='magma')
plt.title('MTTR per Recovery Mode (ms)')
plt.savefig('./docs/images/exp3_recovery_mode.png', bbox_inches='tight')
plt.show()

## Study 4: Group Commit Scaling
Throughput amplification with concurrent writers.

In [ ]:
group = df[df['Event'] == 'WAL_Group_Commit']
plt.figure(figsize=(10, 5))
sns.lineplot(x=group['Count'].astype(int), y=group['Metric'].astype(int), marker='o')
plt.title('Throughput Scaling (Threads vs Ops/s)')
plt.savefig('./docs/images/exp4_group_commit.png', bbox_inches='tight')
plt.show()

## Study 5: Recovery Volume Scaling
Linear time growth for replaying larger WAL volumes.

In [ ]:
scaling = df[df['Event'] == 'WAL_Recovery']
plt.figure(figsize=(10, 5))
sns.lineplot(x=scaling['Count'].astype(int), y=scaling['Metric'].astype(int), marker='s', color='orange')
plt.title('Recovery Time vs Log Volume')
plt.savefig('./docs/images/exp5_scaling.png', bbox_inches='tight')
plt.show()